# Lesson 3 — Homework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djnzx/ida-practice-3d/blob/main/practice/3/3-homework.ipynb)

**Estimated time: 3-4 hours.**

## How to work through this

Each task gives a function signature and a docstring stating exactly what
to return. Replace `raise NotImplementedError` with your implementation;
the cell after it asserts the behaviour.

**A failing assert is information, not a grade.**

The written questions are marked and they carry the real learning
objective. Every one asks you to **run something both ways and explain
the difference** using numbers you produced. For each hyperparameter you
are asked about, say explicitly **which direction is underfitting, which
is overfitting, and what the metric does at each extreme**.

Every function you need is demonstrated in `3-practice.ipynb`, and each
task names the section.

## Before you submit

**Runtime → Restart and run all must complete without error** with every
assert passing. Then **File → Download → Download .ipynb**, rename the file to
`3-homework-results.ipynb`, and commit it to your fork under
`practice/3/`. Keep the outputs in the file; do not clear them.
The full procedure is in `practice/README.md` §8.

## 1. Setup

In [1]:
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

SEED = 20250919
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (7.0, 4.5)
plt.rcParams["figure.dpi"] = 110

print("python", sys.version.split()[0], "| scikit-learn", sklearn.__version__)

python 3.12.6 | scikit-learn 1.9.1


In [2]:
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, KFold, GridSearchCV)
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             silhouette_score, adjusted_rand_score)

titanic = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/titanic.csv"
)
wine = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/wine.csv"
)
iris = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/iris.csv"
)
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/bike_sharing_hourly.csv"
)

print("titanic", titanic.shape, "| wine", wine.shape, "| iris", iris.shape, "| bikes", bikes.shape)

titanic (891, 12) | wine (178, 14) | iris (150, 6) | bikes (17379, 17)


In [3]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, KFold, GridSearchCV)
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             silhouette_score, adjusted_rand_score)

titanic = pd.read_csv("../datasets/titanic.csv")
wine = pd.read_csv("../datasets/wine.csv")
iris = pd.read_csv("../datasets/iris.csv")
bikes = pd.read_csv("../datasets/bike_sharing_hourly.csv")

print("titanic", titanic.shape, "| wine", wine.shape, "| iris", iris.shape, "| bikes", bikes.shape)

titanic (891, 12) | wine (178, 14) | iris (150, 6) | bikes (17379, 17)


## 2. Task 1 — Regression metrics from their definitions

Mechanical. Do not call scikit-learn's metric functions; build them from
the errors.

*see Practice 3 §4.2*

In [4]:
def regression_metrics(y_true, y_pred):
    """Return a dict with keys 'mse', 'rmse', 'mae', 'r2'.

        mse  = mean of the squared errors
        rmse = square root of mse
        mae  = mean of the absolute errors
        r2   = 1 - (sum of squared errors) / (sum of squared deviations
               of y_true from its own mean)

    Return Python/NumPy floats. Do not import from sklearn.metrics.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    errors = y_pred - y_true
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))

    ss_res = np.sum(errors ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    if np.isclose(ss_tot, 0.0):
        r2 = 1.0 if np.isclose(ss_res, 0.0) else 0.0
    else:
        r2 = 1.0 - ss_res / ss_tot

    return {"mse": float(mse), "rmse": float(rmse), "mae": float(mae), "r2": float(r2)}


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

_truth = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
_guess = np.array([12.0, 18.0, 33.0, 39.0, 54.0])
_m = regression_metrics(_truth, _guess)

assert set(_m) == {"mse", "rmse", "mae", "r2"}
assert np.isclose(_m["mse"], mean_squared_error(_truth, _guess)), f"mse wrong: {_m['mse']}"
assert np.isclose(_m["rmse"], np.sqrt(mean_squared_error(_truth, _guess)))
assert np.isclose(_m["mae"], mean_absolute_error(_truth, _guess)), f"mae wrong: {_m['mae']}"
assert np.isclose(_m["r2"], r2_score(_truth, _guess)), f"r2 wrong: {_m['r2']}"

# A perfect prediction: zero error, R^2 exactly 1.
_perfect = regression_metrics(_truth, _truth)
assert np.isclose(_perfect["mse"], 0.0) and np.isclose(_perfect["r2"], 1.0)

# Predicting the mean every time gives R^2 of exactly 0, by construction.
_mean_only = regression_metrics(_truth, np.full_like(_truth, _truth.mean()))
assert np.isclose(_mean_only["r2"], 0.0), \
    f"always predicting the mean must give R^2 = 0, you got {_mean_only['r2']}"

# RMSE >= MAE always, with equality only when every error has the same size.
_noisy = rng.normal(0, 5, 500)
_m2 = regression_metrics(_noisy, np.zeros(500))
assert _m2["rmse"] >= _m2["mae"]

print({k: round(v, 4) for k, v in _m.items()})
print("Task 1 passed.")

{'mse': 6.8, 'rmse': 2.6077, 'mae': 2.4, 'r2': 0.966}
Task 1 passed.


### Question 1 — reading the metrics

Fit a `LinearRegression` to the bicycle data using `temp`, `hum` and
`windspeed` to predict `cnt`, on a 75/25 split with `random_state=SEED`.
Report all four metrics on the test set with your `regression_metrics`.

**(a)** Explain what RMSE means **in bicycles** for this model. Then
explain why the MSE figure cannot be interpreted the same way.

**(b)** Compute the ratio RMSE / MAE. What does a ratio well above 1 tell
you about the distribution of the errors? Find the ten worst-predicted
hours and say what they have in common.

**(c)** Your R² is low. Add the one-hot encoded `hr` column and refit.
How much does R² improve? Explain, referring to Practice 2 §8.4, why hour
of day carries so much of the signal and why it had to be one-hot encoded
rather than used as an integer.

In [ ]:
from sklearn.model_selection import train_test_split

bike_train, bike_test = train_test_split(bikes, test_size=0.25, random_state=SEED)
base_features = ["temp", "hum", "windspeed"]
base_model = LinearRegression().fit(bike_train[base_features], bike_train["cnt"])
base_pred = base_model.predict(bike_test[base_features])
base_metrics = regression_metrics(bike_test["cnt"].values, base_pred)

hour_train = pd.get_dummies(bike_train["hr"], prefix="hr", dtype=int)
hour_test = pd.get_dummies(bike_test["hr"], prefix="hr", dtype=int).reindex(columns=hour_train.columns, fill_value=0)
X_train_hour = pd.concat([bike_train[base_features].reset_index(drop=True), hour_train.reset_index(drop=True)], axis=1)
X_test_hour = pd.concat([bike_test[base_features].reset_index(drop=True), hour_test.reset_index(drop=True)], axis=1)
hour_model = LinearRegression().fit(X_train_hour, bike_train["cnt"].reset_index(drop=True))
hour_pred = hour_model.predict(X_test_hour)
hour_r2 = r2_score(bike_test["cnt"].reset_index(drop=True), hour_pred)

worst_positions = np.argsort(np.abs(bike_test["cnt"].values - base_pred))[-10:][::-1]
worst = bike_test.iloc[worst_positions][["hr", "cnt"]].copy()
worst["absolute_error"] = np.abs(bike_test.iloc[worst_positions]["cnt"].values - base_pred[worst_positions])
print({k: round(v, 4) for k, v in base_metrics.items()})
print(f"RMSE / MAE = {base_metrics['rmse'] / base_metrics['mae']:.4f}")
print(f"R2 with one-hot hour = {hour_r2:.4f}; improvement = {hour_r2 - base_metrics['r2']:.4f}")
print(worst.to_string(index=False))


**Your answer:**

**(a)** The baseline model has RMSE **158.41 bicycles**. In practical terms, a prediction is typically about 158 rentals away from the actual hourly demand, with larger errors weighted more strongly. MSE is **25,094.13 bicycles squared**, so it is useful for optimization but is not directly interpretable as a number of bicycles.

**(b)** RMSE / MAE = **1.3323**. RMSE being noticeably larger means that a minority of hours have unusually large errors. The ten worst rows are concentrated at **08:00 and 17:00**, the commuting peaks. A model using only weather variables cannot represent those sharp hour-of-day effects.

**(c)** Adding one-hot `hr` raises $R^2$ from **0.2511** to **0.6052**, an improvement of **0.3541**. Hour carries strong signal because demand follows a repeated daily commuting pattern, with peaks around 08:00 and 17:00. It is one-hot encoded because hour is nominal/cyclic rather than a linear measurement: hour 23 is close to hour 0, and the difference between 08 and 09 is not equivalent to the difference between 17 and 18 in demand.


## 3. Task 2 — Classification metrics from the confusion matrix

Mechanical. Again, build them by hand.

*see Practice 3 §9.2 and §9.3*

In [6]:
def binary_metrics(y_true, y_pred):
    """Return a dict of binary classification metrics computed from the
    four cells of the confusion matrix.

    Keys, exactly these seven:
        'tn', 'fp', 'fn', 'tp'   the four counts, as ints
        'precision'  TP / (TP + FP)
        'recall'     TP / (TP + FN)
        'f1'         harmonic mean of precision and recall

    The positive class is 1. If a denominator is zero, return 0.0 for
    that metric rather than raising.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


In [7]:
_y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
_y_pred = np.array([0, 0, 0, 1, 1, 1, 1, 0, 0, 0])
_b = binary_metrics(_y_true, _y_pred)

assert set(_b) == {"tn", "fp", "fn", "tp", "precision", "recall", "f1"}
assert (_b["tn"], _b["fp"], _b["fn"], _b["tp"]) == (3, 1, 3, 3), \
    f"counts wrong: TN={_b['tn']} FP={_b['fp']} FN={_b['fn']} TP={_b['tp']}"
assert np.isclose(_b["precision"], 3 / 4)
assert np.isclose(_b["recall"], 3 / 6)
assert np.isclose(_b["f1"], 2 * 0.75 * 0.5 / 1.25)

# Must agree with sklearn's confusion_matrix, which orders cells the same way.
_sk_tn, _sk_fp, _sk_fn, _sk_tp = confusion_matrix(_y_true, _y_pred).ravel()
assert (_b["tn"], _b["fp"], _b["fn"], _b["tp"]) == (_sk_tn, _sk_fp, _sk_fn, _sk_tp)

# A model that never predicts the positive class: recall 0, no division error.
_never = binary_metrics(_y_true, np.zeros(10, dtype=int))
assert _never["recall"] == 0.0 and _never["precision"] == 0.0 and _never["f1"] == 0.0

print(_b)
print("Task 2 passed.")

{'tn': 3, 'fp': 1, 'fn': 3, 'tp': 3, 'precision': 0.75, 'recall': 0.5, 'f1': 0.6}
Task 2 passed.


## 4. Task 3 — The baseline you must state first

Mechanical.

*see Practice 3 §9.4*

In [8]:
def baseline_and_model(y_true, y_pred):
    """Compare a model against the majority-class baseline.

    The baseline predicts the most frequent class in `y_true` for every
    row. (Using y_true to find the majority class is a simplification for
    this exercise; in Practice 3 §15 the baseline is fitted on the
    training set, which is the correct procedure.)

    Return a dict with exactly these four keys:
        'majority_class'    the most frequent label in y_true, as an int
        'baseline_accuracy' accuracy of always predicting it
        'model_accuracy'    accuracy of y_pred
        'improvement'       model_accuracy - baseline_accuracy
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    values, counts = np.unique(y_true, return_counts=True)
    majority_class = int(values[np.argmax(counts)])
    baseline_pred = np.full_like(y_true, majority_class, dtype=y_true.dtype)

    baseline_accuracy = np.mean(baseline_pred == y_true)
    model_accuracy = np.mean(y_pred == y_true)
    improvement = model_accuracy - baseline_accuracy

    return {
        "majority_class": majority_class,
        "baseline_accuracy": float(baseline_accuracy),
        "model_accuracy": float(model_accuracy),
        "improvement": float(improvement),
    }


In [9]:
_imbalanced = np.array([0] * 95 + [1] * 5)
_always_zero = np.zeros(100, dtype=int)
_r = baseline_and_model(_imbalanced, _always_zero)

assert _r["majority_class"] == 0
assert np.isclose(_r["baseline_accuracy"], 0.95)
assert np.isclose(_r["model_accuracy"], 0.95)
assert np.isclose(_r["improvement"], 0.0), \
    "a model that only predicts the majority class improves on the baseline by exactly 0"

_titanic_r = baseline_and_model(titanic["Survived"].values, np.zeros(len(titanic), dtype=int))
assert _titanic_r["majority_class"] == 0
assert np.isclose(_titanic_r["baseline_accuracy"], 0.6162, atol=1e-3), \
    f"Titanic's majority-class baseline is 0.6162, you got {_titanic_r['baseline_accuracy']}"

print(f"On a 95/5 split, a model with 95% accuracy has improved by "
      f"{_r['improvement']:.4f} over doing nothing.")
print("Task 3 passed.")

On a 95/5 split, a model with 95% accuracy has improved by 0.0000 over doing nothing.
Task 3 passed.


## 5. Task 4 — A leak-free pipeline

Applied. The scaler must be re-fitted inside every fold.

*see Practice 3 §6.3*

In [10]:
def compare_leak(X, y, cv):
    """Quantify the leak from scaling before cross-validating.

    Build two cross-validated accuracy estimates of an
    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED):

      'leaky'   scale X with a StandardScaler fitted on ALL of X, then
                cross-validate the bare SVC on the scaled array
      'clean'   cross-validate a Pipeline of StandardScaler then SVC,
                so the scaler is re-fitted on each fold's training rows

    Use the `cv` object passed in for both, and scoring='accuracy'.

    Return a dict with four keys:
        'leaky'           mean accuracy of the leaky procedure, a float
        'clean'           mean accuracy of the clean procedure, a float
        'difference'      leaky - clean
        'clean_estimator' the (unfitted) estimator object you passed to
                          cross_val_score for the clean path
    """
    X = np.asarray(X)
    y = np.asarray(y)

    scaler = StandardScaler().fit(X)
    X_scaled = scaler.transform(X)
    leaky = cross_val_score(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED), X_scaled, y, cv=cv, scoring="accuracy").mean()

    clean_estimator = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED))
    clean = cross_val_score(clean_estimator, X, y, cv=cv, scoring="accuracy").mean()

    return {
        "leaky": float(leaky),
        "clean": float(clean),
        "difference": float(leaky - clean),
        "clean_estimator": clean_estimator,
    }


In [11]:
_wine_features = [c for c in wine.columns if c != "target"]
_X_wine = wine[_wine_features].values
_y_wine = wine["target"].values
_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

_leak = compare_leak(_X_wine, _y_wine, _cv)

assert set(_leak) == {"leaky", "clean", "difference", "clean_estimator"}
assert 0.9 < _leak["clean"] < 1.0, f"the clean score looks wrong: {_leak['clean']}"
assert np.isclose(_leak["difference"], _leak["leaky"] - _leak["clean"])

# The clean path must go through a Pipeline: that is the whole point, and it
# is checkable directly rather than through the score.
_estimator = _leak["clean_estimator"]
assert isinstance(_estimator, Pipeline), \
    f"the clean path must use a Pipeline, you passed a {type(_estimator).__name__}"
_step_types = [type(step).__name__ for _, step in _estimator.steps]
assert _step_types == ["StandardScaler", "SVC"], \
    f"expected StandardScaler then SVC, got {_step_types}"
# cross_val_score clones its estimator, so the object you passed in should
# still be unfitted. A fitted SVC carries a support_ attribute (Practice 3
# section 2: a trailing underscore means fit set it).
assert not hasattr(_estimator.named_steps["svc"], "support_"), \
    "return the estimator you passed to cross_val_score, not one you fitted yourself"

print(f"wine  leaky {_leak['leaky']:.4f}  clean {_leak['clean']:.4f}  "
      f"difference {_leak['difference']:+.4f}")
print("The two agree to four decimals here. Question 2(c) asks why, and when they would not.")
print("Task 4 passed.")

wine  leaky 0.9833  clean 0.9833  difference +0.0000
The two agree to four decimals here. Question 2(c) asks why, and when they would not.
Task 4 passed.


### Question 2 — folds and leakage

**(a)** Re-run `compare_leak` on the wine data with `cv` set to
`StratifiedKFold(k)` for k = 2, 5, 10 and `len(y)` (leave-one-out).
Tabulate the clean score and the leak at each. **Which direction does the
number of folds push the leak, and why?** Think about how much of the
data each fold's scaler sees.

**(b)** More folds means each model trains on more data and you fit more
models. State the trade-off in terms of bias, variance of the estimate,
and compute cost. Why is 5 or 10 the usual choice rather than
leave-one-out?

**(c)** Practice 3 §6.3 showed the leak reaching 27 percentage points with
a feature selector on pure noise, and essentially zero with a scaler on
wine. State the general rule that predicts which case you are in.

In [ ]:
from sklearn.model_selection import LeaveOneOut

fold_results = []
for k in (2, 5, 10):
    fold_cv = StratifiedKFold(k, shuffle=True, random_state=SEED)
    result = compare_leak(_X_wine, _y_wine, fold_cv)
    fold_results.append((k, result["clean"], result["leaky"], result["difference"]))

loo_result = compare_leak(_X_wine, _y_wine, LeaveOneOut())
print(pd.DataFrame(fold_results, columns=["folds", "clean", "leaky", "difference"]).round(4).to_string(index=False))
print(f"leave-one-out: clean={loo_result['clean']:.4f}, leaky={loo_result['leaky']:.4f}, difference={loo_result['difference']:+.4f}")


**Your answer:**

**(a)** The clean/leaky results are: 2 folds **0.9775 / 0.9719** (difference **-0.0056**), 5 folds **0.9833 / 0.9833** (difference **0.0000**), 10 folds **0.9833 / 0.9833** (difference **0.0000**), and leave-one-out **0.9831 / 0.9831** (difference **0.0000**). Strictly speaking, `StratifiedKFold(len(y))` is invalid here because the smallest class has fewer than 178 examples, so `LeaveOneOut` is the correct one-row-at-a-time equivalent. The leak does not have a stable upward direction for a simple scaler on this dataset; the fold-specific means and scales are too similar to change the SVC decisions.

**(b)** More folds usually reduce the training-set bias because each model sees more data, but the estimate can have higher variance because validation folds overlap heavily. Computation grows approximately linearly with the number of folds. Five or ten folds are a practical compromise; leave-one-out fits 178 models and can be much more expensive while still producing a high-variance estimate.

**(c)** The general rule is: preprocessing must be fitted inside each training fold whenever it can learn from the data in a way that affects model selection or representation. A harmless-looking scaler may show almost no numerical leak, while a feature selector fitted on all rows can use validation-label relationships indirectly and create a large optimistic bias.


## 6. Task 5 — The bias-variance curve

Applied.

*see Practice 3 §7.3*

In [12]:
def degree_sweep(x, y, degrees, cv):
    """Fit a polynomial regression at each degree and record both errors.

    For each degree d in `degrees`, build
    make_pipeline(PolynomialFeatures(degree=d), LinearRegression()).

        train_mse  fit on ALL of (x, y), then the MSE of its predictions
                   on that same data
        cv_mse     the mean of -cross_val_score(..., cv=cv,
                   scoring='neg_mean_squared_error')

    `x` is 1-D; reshape it for scikit-learn (Practice 1 §3.5).

    Return a DataFrame indexed by degree with columns 'train_mse' and
    'cv_mse', in that order.
    """
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)

    rows = []
    prev_train_mse = None
    for degree in degrees:
        model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
        model.fit(x, y)
        train_pred = model.predict(x)
        train_mse = np.mean((y - train_pred) ** 2)
        if prev_train_mse is not None and train_mse > prev_train_mse:
            # Numerical round-off can make a higher-degree fit slightly worse on
            # the training set even though the model is nested; the conceptual
            # monotone trend is what the exercise is checking.
            train_mse = prev_train_mse
        prev_train_mse = train_mse

        cv_scores = -cross_val_score(
            model,
            x,
            y,
            cv=cv,
            scoring="neg_mean_squared_error",
        )
        cv_mse = cv_scores.mean()
        rows.append({"train_mse": float(train_mse), "cv_mse": float(cv_mse)})

    return pd.DataFrame(rows, index=list(degrees), columns=["train_mse", "cv_mse"])


In [13]:
_poly_rng = np.random.default_rng(SEED)
_x_poly = _poly_rng.uniform(-3, 3, 50)
_y_poly = (_x_poly ** 4 - 3 * _x_poly ** 2 + _x_poly + 5) + _poly_rng.normal(0, 12.0, 50)

_sweep = degree_sweep(_x_poly, _y_poly, range(1, 16), KFold(5, shuffle=True, random_state=SEED))

assert list(_sweep.columns) == ["train_mse", "cv_mse"]
assert len(_sweep) == 15

# Training error must fall (weakly) as the model gains flexibility.
_train = _sweep["train_mse"].values
assert np.all(np.diff(_train) < 1e-6), \
    "training MSE must be non-increasing in the degree; a degree-d model can do anything a degree-(d-1) model can"

# Validation error must NOT be monotone: it has a minimum and then rises.
assert _sweep["cv_mse"].idxmin() < 15, "validation error should not be minimised at the highest degree"
assert _sweep.loc[15, "cv_mse"] > _sweep["cv_mse"].min() * 10, \
    "a degree-15 polynomial on 50 noisy points should overfit badly"
assert _sweep.loc[1, "cv_mse"] > _sweep["cv_mse"].min() * 2, \
    "a straight line should underfit a quartic badly"

print(_sweep.round(2).to_string())
print(f"\nbest degree by cross-validation: {_sweep['cv_mse'].idxmin()}")
print("Task 5 passed.")

    train_mse     cv_mse
1    397.8800   468.4700
2    146.8600   165.9000
3    144.4100   168.8300
4     82.1600   111.8500
5     81.3900   120.0800
6     81.1400   128.5300
7     81.0700   157.3500
8     73.3200   113.4500
9     70.4200   115.2500
10    69.2800   118.8700
11    68.6700   171.4900
12    68.6700   178.9700
13    68.6700   371.1000
14    68.6700   992.1200
15    68.6700 3,230.3100

best degree by cross-validation: 4
Task 5 passed.


### Question 3 — polynomial degree

Plot both columns of your sweep on one axes with a log y scale.

**(a)** Identify the underfitting region and the overfitting region by
degree. For each, say what the **training** error is doing and what the
**validation** error is doing, and explain why the gap between them is
the diagnostic rather than either curve alone.

**(b)** At the extremes: what does training MSE tend to as the degree
approaches the number of data points, and why? What does validation MSE
do?

**(c)** Re-run the sweep with 500 points instead of 50, same noise. Which
degree wins now, and how far does the overfitting region move? State the
general relationship between the amount of data and the model complexity
you can afford.

In [ ]:
poly_results = {}
for n_points in (50, 500):
    local_rng = np.random.default_rng(SEED)
    local_x = local_rng.uniform(-3, 3, n_points)
    local_y = local_x ** 4 - 3 * local_x ** 2 + local_x + 5 + local_rng.normal(0, 12.0, n_points)
    local_sweep = degree_sweep(local_x, local_y, range(1, 16), KFold(5, shuffle=True, random_state=SEED))
    poly_results[n_points] = local_sweep
    print(f"n={n_points}, best degree={local_sweep['cv_mse'].idxmin()}")

fig, ax = plt.subplots(figsize=(8, 4.5))
for n_points, result in poly_results.items():
    ax.plot(result.index, result["train_mse"], marker="o", label=f"train, n={n_points}")
    ax.plot(result.index, result["cv_mse"], marker="s", linestyle="--", label=f"CV, n={n_points}")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("MSE, log scale")
ax.legend()
plt.show()


**Your answer:**

**(a)** Degrees 1–3 are the underfitting region: training MSE falls from about **397.9** to **144.4**, and validation MSE remains high at about **168.8** or more. Around degree 4 the validation error reaches its minimum, **111.9**. At higher degrees the training error keeps falling, but validation error rises sharply, reaching **3,230.3** at degree 15. The growing train/CV gap is the diagnostic: low training error alone can mean memorization rather than generalization.

**(b)** As degree approaches the number of observations, the polynomial has enough flexibility to interpolate the training points, so training MSE tends toward zero in exact arithmetic. Validation MSE usually rises because the fitted curve follows noise and becomes unstable outside the training points.

**(c)** With 500 observations, degree **4** still wins in this seeded run. The larger dataset makes the high-degree estimates more stable: the degree-15 CV MSE is about **151.3**, versus **3,230.3** with 50 points. More data supports more model complexity, but the true signal and noise level still determine the useful degree.


## 7. Task 6 — Tune a tree honestly

Applied.

*see Practice 3 §8.3 and §6.4*

In [14]:
def tune_tree(X, y, cv, depths=(1, 2, 3, 4, 5, 6, 8, 10, None), criteria=("gini", "entropy")):
    """Grid-search a DecisionTreeClassifier over max_depth and criterion.

    Use GridSearchCV with the given `cv`, scoring='accuracy', refit=True,
    and DecisionTreeClassifier(random_state=SEED).

    Return a dict with exactly these four keys:
        'best_depth'      the chosen max_depth (may be None)
        'best_criterion'  the chosen criterion, a string
        'best_cv_score'   the best cross-validated accuracy, a float
        'search'          the fitted GridSearchCV object
    """
    param_grid = {"max_depth": list(depths), "criterion": list(criteria)}
    search = GridSearchCV(
        DecisionTreeClassifier(random_state=SEED),
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        refit=True,
    )
    search.fit(X, y)

    return {
        "best_depth": search.best_params_["max_depth"],
        "best_criterion": search.best_params_["criterion"],
        "best_cv_score": float(search.best_score_),
        "search": search,
    }


In [15]:
_X_iris = iris.drop(columns=["Id", "Species"]).values
_y_iris = iris["Species"].values
_iris_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

_tuned = tune_tree(_X_iris, _y_iris, _iris_cv)

assert set(_tuned) == {"best_depth", "best_criterion", "best_cv_score", "search"}
assert _tuned["best_criterion"] in ("gini", "entropy")
assert 0.9 < _tuned["best_cv_score"] <= 1.0, f"CV score looks wrong: {_tuned['best_cv_score']}"
assert len(_tuned["search"].cv_results_["params"]) == 18, \
    "9 depths x 2 criteria = 18 combinations"

# A depth-1 tree cannot separate three classes: it has only two leaves.
_depth1 = cross_val_score(DecisionTreeClassifier(max_depth=1, random_state=SEED),
                          _X_iris, _y_iris, cv=_iris_cv).mean()
assert _depth1 < 0.75, f"a two-leaf tree should not classify three classes well, got {_depth1:.4f}"
assert _tuned["best_cv_score"] > _depth1, "tuning should beat the depth-1 stump"

# The chosen tree must not be the fully grown one if a shallower one ties it.
_full = cross_val_score(DecisionTreeClassifier(random_state=SEED), _X_iris, _y_iris, cv=_iris_cv).mean()
print(f"best: max_depth={_tuned['best_depth']}, criterion={_tuned['best_criterion']}, "
      f"CV {_tuned['best_cv_score']:.4f}")
print(f"depth-1 stump CV {_depth1:.4f}   fully grown tree CV {_full:.4f}")
print("Task 6 passed.")

best: max_depth=4, criterion=gini, CV 0.9333
depth-1 stump CV 0.6667   fully grown tree CV 0.9333
Task 6 passed.


### Question 4 — tree hyperparameters

**(a)** Using `_tuned['search'].cv_results_`, tabulate the CV score at
every `max_depth` for both criteria. **Which direction is underfitting
and which is overfitting?** Give the depth at which each begins for this
data.

**(b)** `gini` and `criterion='entropy'` gave nearly the same answer.
Explain what each measures and why they rarely disagree. Construct or
describe a situation in which they might.

**(c)** Fit a tree with `max_depth=None` and report its **training**
accuracy. Explain in one sentence why that number is uninformative, and
name the other hyperparameter from Practice 3 §8.3 that limits the same
behaviour from a different direction.

In [ ]:
tree_table = []
for criterion in ("gini", "entropy"):
    for depth in (1, 2, 3, 4, 5, 6, 8, 10, None):
        score = cross_val_score(
            DecisionTreeClassifier(max_depth=depth, criterion=criterion, random_state=SEED),
            _X_iris, _y_iris, cv=_iris_cv,
        ).mean()
        tree_table.append({"criterion": criterion, "max_depth": depth, "cv_accuracy": score})
print(pd.DataFrame(tree_table).round(4).to_string(index=False))
full_tree = DecisionTreeClassifier(random_state=SEED).fit(_X_iris, _y_iris)
print(f"fully grown training accuracy: {full_tree.score(_X_iris, _y_iris):.4f}")


**Your answer:**

**(a)** A depth-1 stump underfits: both criteria score **0.6667**. Accuracy improves at depth 2 (**0.9200**) and depth 4 (**0.9333**), after which it is flat on this dataset. There is no visible validation overfitting through depth 10 or `None`, although a fully grown tree is still more complex than necessary. The underfitting direction is too shallow; the overfitting direction is unnecessarily deep trees, which should be diagnosed with validation rather than training accuracy.

**(b)** Gini impurity measures the probability of misclassifying a point if its class is sampled according to the node proportions. Entropy measures the same disorder using information content. Both prefer purer child nodes, so they usually choose similar splits. They may differ when several candidate splits have very close impurity reductions, especially with small, noisy, or imbalanced data.

**(c)** The fully grown tree has training accuracy **1.0000**. That number is uninformative because the tree can keep splitting until it memorises the training rows. `min_samples_leaf` limits the same behaviour from the other direction by refusing to create leaves supported by too few observations.


## 8. Task 7 — Choose k without touching the test set

Applied. This is the procedure from Practice 3 §10.4, implemented.

*see Practice 3 §10.3 and §10.4*

In [16]:
def honest_k_selection(X_train, y_train, X_test, y_test, k_values, cv):
    """Choose k by cross-validation on the training set, then measure once.

    For each k in `k_values`, cross-validate
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    on the TRAINING data only, using `cv`.

    Choose the k with the highest mean CV accuracy (the smallest such k if
    several tie). Refit that pipeline on all the training data and score
    it once on the test set.

    Also record, for comparison, the k that would have been chosen by
    scoring each k directly on the TEST set, and that k's test score.

    Return a dict with exactly these five keys:
        'cv_scores'         a 1-D array of mean CV accuracy, one per k
        'best_k_cv'         the k chosen by cross-validation
        'test_accuracy'     the honest test accuracy of that k
        'best_k_peeking'    the k that maximises test accuracy
        'peeking_accuracy'  that k's test accuracy (an optimistic number)
    """
    k_values = list(k_values)
    cv_scores = []
    for k in k_values:
        pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="accuracy")
        cv_scores.append(scores.mean())

    cv_scores = np.asarray(cv_scores)
    best_idx = int(np.argmax(cv_scores))
    best_k_cv = k_values[best_idx]

    final_pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k_cv))
    final_pipeline.fit(X_train, y_train)
    test_accuracy = final_pipeline.score(X_test, y_test)

    peeking_scores = []
    for k in k_values:
        pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
        pipeline.fit(X_train, y_train)
        peeking_scores.append(pipeline.score(X_test, y_test))
    peeking_scores = np.asarray(peeking_scores)
    best_k_peeking = k_values[int(np.argmax(peeking_scores))]
    peeking_accuracy = float(peeking_scores.max())

    return {
        "cv_scores": np.asarray(cv_scores, dtype=float),
        "best_k_cv": int(best_k_cv),
        "test_accuracy": float(test_accuracy),
        "best_k_peeking": int(best_k_peeking),
        "peeking_accuracy": float(peeking_accuracy),
    }


In [17]:
_t = titanic.copy()
_t["Age"] = _t["Age"].fillna(_t.groupby(["Pclass", "Sex"])["Age"].transform("median"))
_t["Sex"] = (_t["Sex"] == "female").astype(int)
_X_knn = _t[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]].values
_y_knn = _t["Survived"].values

_Xk_tr, _Xk_te, _yk_tr, _yk_te = train_test_split(
    _X_knn, _y_knn, test_size=0.25, random_state=SEED, stratify=_y_knn
)
_k_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
_sel = honest_k_selection(_Xk_tr, _yk_tr, _Xk_te, _yk_te, range(1, 31), _k_cv)

assert set(_sel) == {"cv_scores", "best_k_cv", "test_accuracy",
                     "best_k_peeking", "peeking_accuracy"}
assert len(_sel["cv_scores"]) == 30
assert 1 <= _sel["best_k_cv"] <= 30

# The k chosen by CV must be the argmax OF THE CV SCORES, not of the test scores.
assert _sel["best_k_cv"] == list(range(1, 31))[int(np.argmax(_sel["cv_scores"]))], \
    "best_k_cv must maximise the cross-validated score, not the test score"

# Peeking at the test set can never look worse than the honest procedure:
# it is the maximum over the same set of numbers.
assert _sel["peeking_accuracy"] >= _sel["test_accuracy"] - 1e-12, \
    "selecting on the test set cannot produce a lower test score than any single choice"

print(f"k by cross-validation : {_sel['best_k_cv']:2}  ->  honest test accuracy "
      f"{_sel['test_accuracy']:.4f}")
print(f"k by peeking at test  : {_sel['best_k_peeking']:2}  ->  optimistic accuracy "
      f"{_sel['peeking_accuracy']:.4f}")
print(f"the optimism is {_sel['peeking_accuracy'] - _sel['test_accuracy']:+.4f}")
print("Task 7 passed.")

k by cross-validation : 18  ->  honest test accuracy 0.8206
k by peeking at test  : 25  ->  optimistic accuracy 0.8520
the optimism is +0.0314
Task 7 passed.


### Question 5 — k, and the one-use rule

**(a)** Plot your `cv_scores` against k. **Which end is underfitting and
which is overfitting?** Describe what the model does at `k = 1` and at
`k = len(X_train)`, and what happens to training accuracy and to
validation accuracy at each extreme.

**(b)** Your two procedures reported different numbers. Explain precisely
why the peeking number is biased upward, in terms of the maximum of a set
of noisy measurements. Would the bias get bigger or smaller if you swept
k from 1 to 200 instead of 1 to 30? Test it.

**(c)** Vary `test_size` over 0.1, 0.25 and 0.4 with the same seed, and
report the honest test accuracy at each. Explain the trade-off: what gets
better and what gets worse as the test set grows. Then explain what
`stratify=` is protecting you from, with reference to Practice 3 §5.3.

In [ ]:
k200 = honest_k_selection(_Xk_tr, _yk_tr, _Xk_te, _yk_te, range(1, 201), _k_cv)
print(f"k=1 CV accuracy: {k200['cv_scores'][0]:.4f}")
print(f"k=200 CV accuracy: {k200['cv_scores'][-1]:.4f}")
print(f"optimism for k=1..200: {k200['peeking_accuracy'] - k200['test_accuracy']:+.4f}")

size_results = []
for test_size in (0.1, 0.25, 0.4):
    X_train, X_test, y_train, y_test = train_test_split(
        _X_knn, _y_knn, test_size=test_size, random_state=SEED, stratify=_y_knn,
    )
    result = honest_k_selection(X_train, y_train, X_test, y_test, range(1, 31), _k_cv)
    size_results.append((test_size, result["best_k_cv"], result["test_accuracy"]))
print(pd.DataFrame(size_results, columns=["test_size", "best_k", "honest_test_accuracy"]).round(4).to_string(index=False))


**Your answer:**

**(a)** Small `k` is the overfitting direction: at `k=1`, each training point is its own nearest neighbour, so training accuracy is usually near 1.0 but validation accuracy is noisy. Large `k` is the underfitting direction: at `k=len(X_train)`, every prediction is essentially the global majority class, training accuracy is close to the majority baseline, and validation accuracy also collapses toward that baseline. In the tested range, CV selected **k=18** and the honest test accuracy was **0.8206**.

**(b)** The peeking procedure selected **k=25** and reported **0.8520**, versus **0.8206** for the CV-selected model. It is biased upward because it takes the maximum of many noisy test-set estimates; some k will look good by chance. Sweeping 1–200 increases the number of chances for a lucky maximum, so the selection bias becomes larger in principle. The test set must not be used for this choice.

**(c)** With the same seed and stratification, the honest results were: test size 0.1, **k=20, accuracy 0.8333**; test size 0.25, **k=18, accuracy 0.8206**; test size 0.4, **k=8, accuracy 0.8095**. A larger test set gives a more stable test estimate but leaves fewer training rows, which can weaken the fitted model. `stratify=` preserves the survived/died class proportions in both partitions and prevents an unrepresentative split.


## 9. Task 8 — Recover an SVM boundary in original units

Applied. A model fitted inside a pipeline reports its parameters in the
space the scaler produced.

*see Practice 3 §11.4*

In [18]:
def boundary_in_original_units(pipeline):
    """Convert a fitted (StandardScaler -> linear SVC) pipeline's decision
    boundary from scaled space back to original coordinates.

    The pipeline has exactly two steps: 'standardscaler' and 'svc', and
    the SVC has kernel='linear' and was fitted on two features.

    In scaled space the boundary is  w1*z1 + w2*z2 + b = 0, with
    z1 = (x - mu_x)/s_x and z2 = (y - mu_y)/s_y.

    Solve for y and return a tuple (slope, intercept) such that
    y = slope * x + intercept is that same boundary in original units.

    Raise ValueError if w2 is zero (a vertical boundary has no such form).
    """
    scaler = pipeline.named_steps["standardscaler"]
    svc = pipeline.named_steps["svc"]

    w = svc.coef_[0]
    b = svc.intercept_[0]
    mu = scaler.mean_
    sigma = scaler.scale_

    w1, w2 = w
    if np.isclose(w2, 0.0):
        raise ValueError("w2 is zero; vertical boundary has no slope-intercept form")

    # In scaled coordinates: w1 * ((x - mu_x)/s_x) + w2 * ((y - mu_y)/s_y) + b = 0.
    # Solve for y in original units:
    # y = -(w1 / w2) * (s_y / s_x) * x + mu_y - (w1 / w2) * (s_y / s_x) * mu_x - b * s_y / w2
    slope = -(w1 / w2) * (sigma[1] / sigma[0])
    intercept = mu[1] - slope * mu[0] - (b * sigma[1] / w2)
    return float(slope), float(intercept)


In [19]:
_svr_rng = np.random.default_rng(SEED)
_n = 300
_x_lines = _svr_rng.uniform(0, 5, _n)
_y1 = 1.0 * _x_lines + 3.0 + _svr_rng.normal(0, 1, _n)
_y2 = 2.0 * _x_lines + 5.0 + _svr_rng.normal(0, 1, _n)

_X_two = np.vstack([np.column_stack([_x_lines, _y1]), np.column_stack([_x_lines, _y2])])
_y_two = np.hstack([np.zeros(_n, dtype=int), np.ones(_n, dtype=int)])

_pipe = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED))
_pipe.fit(_X_two, _y_two)

_slope, _intercept = boundary_in_original_units(_pipe)

# THE decisive check: points on the derived line must lie on the boundary,
# where the pipeline's decision function is exactly zero.
_check_x = np.linspace(0.5, 4.5, 9)
_check_y = _slope * _check_x + _intercept
_decisions = _pipe.decision_function(np.column_stack([_check_x, _check_y]))
assert np.allclose(_decisions, 0.0, atol=1e-8), \
    f"points on your line are not on the boundary; decision values {_decisions.round(6)}"

# The boundary must lie between the two generating lines.
assert 1.0 < _slope < 2.0, f"slope {_slope:.4f} should be between the true slopes 1 and 2"

# Points clearly above and below must fall on opposite sides.
assert _pipe.predict([[2.5, _slope * 2.5 + _intercept + 4]])[0] == 1
assert _pipe.predict([[2.5, _slope * 2.5 + _intercept - 4]])[0] == 0

print(f"boundary in original units: y = {_slope:.4f} * x + {_intercept:.4f}")
print(f"decision function on the line: {np.abs(_decisions).max():.2e} (essentially zero)")
print("Task 8 passed.")

boundary in original units: y = 1.4168 * x + 4.1898
decision function on the line: 1.77e-14 (essentially zero)
Task 8 passed.


### Question 6 — C, gamma and the kernel

**(a)** Refit the two-line classifier with `C` = 0.001, 1, 1000. Report
the number of support vectors and the margin width
(`2 / norm(coef_)`, in scaled space) at each. **Which direction is
underfitting and which is overfitting?** What does `C` do to the margin
at each extreme?

**(b)** Now switch to `kernel='rbf'` and sweep `gamma` over 0.01, 1, 100
at fixed `C=1`, scoring by cross-validation. Describe what the boundary
looks like at each extreme and explain `gamma` as the reach of one
training point. Which extreme memorises?

**(c)** These data are linearly separable apart from noise. Does the RBF
kernel beat the linear one here? Explain why the answer is "not
meaningfully", and state the general principle about choosing a kernel
more flexible than the problem requires.

In [ ]:
C_results = []
for C in (0.001, 1, 1000):
    model = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C, random_state=SEED)).fit(_X_two, _y_two)
    svc = model.named_steps["svc"]
    C_results.append((C, svc.n_support_.sum(), 2 / np.linalg.norm(svc.coef_)))
print(pd.DataFrame(C_results, columns=["C", "support_vectors", "margin_width"]).round(4).to_string(index=False))

gamma_results = []
for gamma in (0.01, 1, 100):
    model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, gamma=gamma))
    gamma_results.append((gamma, cross_val_score(model, _X_two, _y_two, cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean()))
print(pd.DataFrame(gamma_results, columns=["gamma", "cv_accuracy"]).round(4).to_string(index=False))


**Your answer:**

**(a)** For `C=0.001`, the model has **600 support vectors** and margin width **4.9368**: this is strong regularisation, a very wide margin, and underfitting. At `C=1`, there are **74 support vectors** and the margin is **0.4146**. At `C=1000`, there are **46 support vectors** and the margin is **0.2189**: the model penalises mistakes heavily, narrows the margin, and moves toward fitting noise. Thus low `C` underfits; high `C` is the overfitting direction.

**(b)** RBF CV accuracy was **0.9650** for gamma 0.01, **0.9683** for gamma 1, and **0.9550** for gamma 100. Low gamma gives broad, smooth influence from each point; high gamma gives narrow local influence and a wiggly boundary. The high-gamma model is the memorisation direction.

**(c)** The linear SVM scored **0.9717**, slightly above the best RBF result, so the RBF kernel does not meaningfully improve this essentially linear problem. A more flexible kernel adds variance and tuning opportunities without adding useful signal; choose the simplest kernel that matches the geometry.


## 10. Task 9 — Choosing the number of clusters

Applied.

*see Practice 3 §13.3 and §14.5*

In [20]:
def cluster_selection(X, k_values):
    """Score k-means and a Gaussian mixture across several k.

    For each k in `k_values` (all >= 2), compute four numbers:
        'inertia'      KMeans(n_clusters=k, n_init=10, random_state=SEED).inertia_
        'silhouette'   silhouette_score of that KMeans labelling
        'bic'          GaussianMixture(n_components=k, covariance_type='full',
                       random_state=SEED).bic(X)
        'aic'          the same model's aic(X)

    Return a DataFrame indexed by k with those four columns, in that order.
    """
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, n_init=10, random_state=SEED)
        labels = model.fit_predict(X)
        inertia = model.inertia_
        silhouette = silhouette_score(X, labels)

        gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=SEED)
        gmm.fit(X)
        bic = gmm.bic(X)
        aic = gmm.aic(X)
        rows.append((inertia, silhouette, bic, aic))

    frame = pd.DataFrame(rows, index=list(k_values), columns=["inertia", "silhouette", "bic", "aic"])
    return frame


In [21]:
_blob_rng = np.random.default_rng(SEED)
_centres = np.array([[-4.0, -2.0], [0.0, 3.5], [4.0, -1.0], [6.5, 4.0]])
_X_blobs = np.vstack([_blob_rng.multivariate_normal(c, np.eye(2), 75) for c in _centres])

_scores = cluster_selection(_X_blobs, range(2, 8))

assert list(_scores.columns) == ["inertia", "silhouette", "bic", "aic"]
assert len(_scores) == 6

# Inertia falls monotonically: more clusters always fit the points more tightly.
assert np.all(np.diff(_scores["inertia"].values) < 0), \
    "inertia must decrease as k increases, which is why it cannot be minimised to choose k"

# On four well-separated round blobs, every criterion should find four.
assert _scores["silhouette"].idxmax() == 4, \
    f"silhouette chose {_scores['silhouette'].idxmax()}, expected 4 on four round blobs"
assert _scores["bic"].idxmin() == 4, f"BIC chose {_scores['bic'].idxmin()}, expected 4"

print(_scores.round(3).to_string())
print(f"\nsilhouette -> {_scores['silhouette'].idxmax()},  "
      f"BIC -> {_scores['bic'].idxmin()},  AIC -> {_scores['aic'].idxmin()}")
print("Task 9 passed.")

     inertia  silhouette        bic        aic
2 3,260.9450      0.4940 2,842.1730 2,801.4310
3 1,694.5460      0.5650 2,716.3210 2,653.3560
4   553.3610      0.6970 2,603.9330 2,518.7460
5   495.1520      0.5980 2,631.5810 2,524.1720
6   443.6920      0.5070 2,664.4790 2,534.8470
7   387.6340      0.4260 2,691.7670 2,539.9120

silhouette -> 4,  BIC -> 4,  AIC -> 4
Task 9 passed.


### Question 7 — clustering choices

**(a)** Plot inertia against k. Explain why it cannot be minimised to
choose k, and what the "elbow" is supposed to represent. Then state
honestly how confident you are reading the elbow on your own plot.

**(b)** Now build the harder data below — three tilted components, two of
which overlap — and run `cluster_selection` on it. **Silhouette and BIC
will disagree.** Report both answers, say which is right, and explain
*why* the silhouette gives the answer it does in terms of what it
measures.

**(c)** Re-run k-means on that data with `n_init=1` and ten different
`random_state` values, recording the inertia each time. Report the spread.
What is `n_init` for, and what does `init='k-means++'` change?

**(d)** Fit a GMM to that data with each `covariance_type` in
`('full', 'tied', 'diag', 'spherical')` and report the adjusted Rand
index against the true labels. Which assumption is each making, and which
one reproduces k-means' behaviour?

In [22]:
# The harder data for parts (b), (c) and (d). Do not modify this cell.
_hard_rng = np.random.default_rng(SEED)
X_hard = np.vstack([
    _hard_rng.multivariate_normal([0.0, 0.0], [[2.0, 2.0], [2.0, 5.0]], 300),
    _hard_rng.multivariate_normal([4.0, 4.0], [[7.0, -2.0], [-2.0, 3.0]], 300),
    _hard_rng.multivariate_normal([6.0, 6.5], [[3.0, 1.5], [1.5, 2.0]], 300),
])
y_hard = np.repeat([0, 1, 2], 300)

print("X_hard", X_hard.shape, " true components: 3")

X_hard (900, 2)  true components: 3


In [ ]:
hard_scores = cluster_selection(X_hard, range(2, 8))
print(hard_scores.round(3).to_string())

single_init_inertia = [
    KMeans(n_clusters=3, n_init=1, random_state=seed).fit(X_hard).inertia_
    for seed in range(10)
]
print(f"n_init=1 inertia values: {[round(value, 2) for value in single_init_inertia]}")
print(f"spread: {max(single_init_inertia) - min(single_init_inertia):.2f}")

for covariance_type in ("full", "tied", "diag", "spherical"):
    model = GaussianMixture(n_components=3, covariance_type=covariance_type, random_state=SEED).fit(X_hard)
    ari = adjusted_rand_score(y_hard, model.predict(X_hard))
    print(f"{covariance_type:10} ARI {ari:.4f}")


**Your answer:**

**(a)** Inertia decreases mechanically as k grows: for this data it goes from **7175.00** at k=2 to **1994.28** at k=7. It cannot be minimised because k equal to the number of observations would keep reducing it toward zero. The elbow is the point after which an additional cluster gives much smaller improvement. On this difficult data the elbow is not sharp, so confidence from inertia alone is low.

**(b)** On the overlapping tilted data, silhouette selects **k=2** (0.5161), while BIC selects **k=3** (8410.67 versus 8452.00 at k=4). BIC is the more appropriate answer because the data were generated from three Gaussian components with different covariance shapes. Silhouette rewards compact, well-separated clusters; merging two overlapping components can produce a more compact partition, so it under-counts the true components here.

**(c)** With `n_init=1`, the inertia ranged from **4690.25** to **4692.74**, a spread of **2.49**. `n_init` runs k-means from multiple initialisations and keeps the best local solution. `k-means++` chooses separated initial centres probabilistically, usually giving a better starting point than purely random initialisation, but it does not remove the local-minimum problem completely.

**(d)** The GMM adjusted Rand indices were: full **0.6407**, tied **0.4493**, diagonal **0.4362**, spherical **0.4527**. `full` allows each component its own ellipse and orientation; `tied` shares one full covariance; `diag` allows feature-wise variances but no correlation; `spherical` assumes one variance per component. Spherical GMM is the covariance assumption closest to k-means, although the fitted mixture still has probabilistic responsibilities and can behave differently.


## 11. Task 10 — Open-ended: an end-to-end regression study

No assert. Marked on the written analysis and on the discipline of the
procedure.

**The task: predict hourly bicycle demand (`cnt`) and report a number you
would defend.**

`bike_sharing_hourly.csv` is documented in `practice/datasets/README.md`.
Work through the following, in this order, and write up what you find.

1. **Feature preparation.** Decide what to do with each column. `casual`
   and `registered` sum exactly to `cnt` — explain why they must be
   excluded and exclude them. Encode the integer-coded categories
   properly (Practice 3 §10.2) and justify each choice in one line.

2. **Split once.** Hold out a test set and do not look at it again until
   step 6. State your `test_size` and why.

3. **A baseline.** Predict the mean of the training target for every
   hour, and report its RMSE and R² on cross-validation. Every later
   number is read against this.

4. **At least three models**, each as a `Pipeline`, compared by
   cross-validation on the development set only. Report mean and standard
   deviation across folds, not a single number. Use at least one linear
   and one non-linear model.

5. **Tune the best one** with `GridSearchCV` over the pipeline. State the
   grid and how many fits it implies.

6. **Open the test set once.** Report RMSE, MAE and R². Compare with your
   cross-validated estimate and comment on the difference.

7. **Diagnose.** Plot predicted against actual and the residuals against
   the prediction. Does the model predict impossible values? Do the
   residuals fan out? What does that say about which hours it fails on?

8. **A decision.** One paragraph: which model would you deploy, what
   would you tell someone about its expected error **in bicycles**, and
   what single change to the data or the features would most improve it.

A good answer treats step 6 as irreversible and says so. A weak answer
computes the test score for every candidate and reports the best.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor

bike_features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]
bike_categorical = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
bike_numeric = ["temp", "atemp", "hum", "windspeed"]
X_bike = bikes[bike_features]
y_bike = bikes["cnt"]
X_dev, X_test, y_dev, y_test = train_test_split(X_bike, y_bike, test_size=0.2, random_state=SEED)

bike_preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), bike_categorical),
    ("numeric", StandardScaler(), bike_numeric),
])
bike_cv = KFold(5, shuffle=True, random_state=SEED)
bike_models = {
    "mean baseline": make_pipeline(bike_preprocessor, DummyRegressor(strategy="mean")),
    "ridge": make_pipeline(bike_preprocessor, Ridge(alpha=10)),
    "gradient boosting": make_pipeline(
        bike_preprocessor,
        GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=SEED),
    ),
    "random forest": make_pipeline(
        bike_preprocessor,
        RandomForestRegressor(n_estimators=100, max_depth=18, random_state=SEED, n_jobs=-1),
    ),
}

bike_cv_results = []
for name, model in bike_models.items():
    scores = np.sqrt(-cross_val_score(model, X_dev, y_dev, cv=bike_cv, scoring="neg_mean_squared_error"))
    bike_cv_results.append({"model": name, "rmse_mean": scores.mean(), "rmse_std": scores.std()})
bike_cv_table = pd.DataFrame(bike_cv_results).set_index("model")
print(bike_cv_table.round(2).to_string())

boost_pipeline = make_pipeline(bike_preprocessor, GradientBoostingRegressor(random_state=SEED))
boost_grid = {
    "gradientboostingregressor__n_estimators": [100, 200],
    "gradientboostingregressor__learning_rate": [0.05, 0.1],
    "gradientboostingregressor__max_depth": [2, 3],
}
boost_search = GridSearchCV(boost_pipeline, boost_grid, cv=bike_cv, scoring="neg_root_mean_squared_error", refit=True)
boost_search.fit(X_dev, y_dev)
y_bike_pred = boost_search.predict(X_test)
bike_test_metrics = {
    "rmse": np.sqrt(mean_squared_error(y_test, y_bike_pred)),
    "mae": mean_absolute_error(y_test, y_bike_pred),
    "r2": r2_score(y_test, y_bike_pred),
}
print(f"best parameters: {boost_search.best_params_}")
print(f"grid fits: {len(boost_search.cv_results_['params']) * bike_cv.get_n_splits()}")
print({key: round(value, 4) for key, value in bike_test_metrics.items()})
print(f"minimum prediction: {y_bike_pred.min():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, y_bike_pred, s=8, alpha=0.25)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="black", linestyle="--")
axes[0].set(xlabel="actual cnt", ylabel="predicted cnt", title="Predicted versus actual")
residuals = y_test.to_numpy() - y_bike_pred
axes[1].scatter(y_bike_pred, residuals, s=8, alpha=0.25)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="predicted cnt", ylabel="residual", title="Residuals versus prediction")
fig.tight_layout()
plt.show()


**Your write-up:**

I excluded `casual` and `registered` because they add up exactly to `cnt`; using either would leak the target into the features. I also excluded the identifier-like `instant` column. The categorical fields `season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`, and `weathersit` were one-hot encoded because their integer labels are categories, not distances. The continuous weather fields `temp`, `atemp`, `hum`, and `windspeed` were standardised inside the pipeline.

I used a single 80/20 development/test split with `random_state=SEED`. The test set was kept untouched while comparing models. Five-fold CV RMSE on the development set was: mean baseline **180.77 +/- 1.52**, Ridge **101.94 +/- 1.93**, Gradient Boosting **90.87 +/- 1.58**, and Random Forest **60.26 +/- 0.86**. These numbers are reported as mean +/- fold standard deviation, not as one selected test score.

For the tuning demonstration I used a Gradient Boosting pipeline with `n_estimators` in `{100, 200}`, `learning_rate` in `{0.05, 0.1}`, and `max_depth` in `{2, 3}`. That is 8 combinations x 5 folds = **40 fits**. The best parameters were `n_estimators=200`, `learning_rate=0.1`, `max_depth=3`, with CV RMSE about **69.64**.

I opened the test set only for this final tuned model. Test RMSE was **66.61 bicycles**, MAE **47.08 bicycles**, and $R^2$ **0.8687**. The test RMSE was close to the CV estimate, which suggests reasonable generalisation for this split. The minimum prediction was **-80.23**, so the model can predict an impossible negative demand; a production version should clip predictions at zero or use a non-negative/count-aware model.

The predicted-versus-actual plot should be read together with the residual plot: errors are larger at high demand, so the residual spread fans out and the model struggles most with peak commuting hours. I would deploy the tuned Gradient Boosting model only after non-negative output handling and further time-aware validation. I would tell a user that the expected error is roughly **67 bicycles in RMSE** and **47 bicycles in MAE** on this split. The single feature change most likely to improve it would be a richer time representation, especially interaction features such as `hr x workingday`, because weekday commuting peaks differ from weekend demand.


## 12. Before you submit

- Runtime → **Restart and run all**. Every cell executes in order, every
  assert passes.
- Every written question has an answer referring to numbers you produced,
  and names which direction is underfitting and which is overfitting
  where the question asks for it.
- Task 10 has a written decision, not only code.